# 12 · DA 단독 실시간 (FastSAM 없이) — 더 빠르고 검출 깔끔

실험 결론(11번 참고): FastSAM은 병목이 아니지만(61ms) **물체 선별을 스스로 못 함**(보드 전체 분할). 반대로 **DA는 혼자서 깔끔하게 물체를 잡음**(솟은 영역). 그래서 FastSAM을 빼면 **더 빠르고(≈2.9fps) 검출도 더 깔끔**.

**흐름:** DA 깊이 → ArUco로 '보드평면 위 높이' 변환 → 높이>임계 = 물체 → 점군으로 원통 적합(수직/누움) → 가상 3D 배치 + 전체 마커 지도.

| | DA만 (이 노트북) | DA+FastSAM (08/09) |
|---|---|---|
| 속도(동기) | **~2.9 fps** | ~1.4 fps |
| 검출 | 깔끔(솟음 기준) | 깔끔 |
| 경계·크기 | 약간 거침(높이 과소↑) | FastSAM으로 정밀 |
| 모델 | 1개(DA) | 2개(DA+FastSAM) |

> DA 후처리(픽셀당 3D)를 GPU(torch)로 이식해 height_map 547→~260ms로 단축한 것도 반영됨.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, depth_volume as dv, live_da as ld, scene3d as s3

OUTPUT_DIR = os.path.join(ROOT, 'output'); SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
CAM_INDEX = 0
pipe = dv.load_depth_model('depth-anything/Depth-Anything-V2-Small-hf', device=0)  # DA만 로드(FastSAM 없음)
print('ready (DA only)')

## 정지영상 미리보기 (카메라 | 가상 3D) + 인터랙티브 뷰어

In [ ]:
cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'snap_raw_*.png'))) or sorted(glob.glob(os.path.join(SCENE_DIR, '*.*')))
img = cv2.imread(cands[-1]); print('대상:', os.path.basename(cands[-1]))
vis, objs, markers = ld.process_frame_da(img, pipe, board, K, dist)
print(f'물체 {len(objs)}개, 마커 {len(markers)}개')
for i, o in enumerate(objs):
    print(f"  #{i} {o['type']:5s} {o['label']}  shape={o['shape']}")
scene = s3.render_virtual_scene(objs, markers=markers, ws=(240, 320))
camh = cv2.resize(vis, (int(vis.shape[1]*scene.shape[0]/vis.shape[0]), scene.shape[0]))
plt.figure(figsize=(18, 7)); plt.imshow(cv2.cvtColor(np.hstack([camh, scene]), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
# 인터랙티브(마우스 회전/확대) — 브라우저로 열림
s3.render_plotly(objs, markers=markers, html_path=os.path.join(OUTPUT_DIR, 'virtual_scene.html'), open_browser=True)
print('인터랙티브: output/virtual_scene.html')

## 실시간 (DA 단독)

보드 위에 물체를 올리면 검출·가상 3D 배치가 실시간(~2.9fps)으로 갱신. **빈 보드 기준(`r`) 불필요** — 깊이가 솟음을 직접 감지.
- **`s`**: 스냅  ·  **`q`**: 종료

In [ ]:
ld.run_live_da(K, dist, board, pipe=pipe, square_len=SQ, cam_index=CAM_INDEX, snapshot_dir=OUTPUT_DIR)
print('종료됨')

## 정리 & 튜닝

- **더 빠름**: FastSAM 제거로 ~2배(1.4→2.9fps), 메모리·로딩도 절약.
- **검출 깔끔**: 솟은 영역만 → 보드 오검출 없음, 선/누움 판별.
- **트레이드오프**: 경계가 높이-블롭이라 지름·바닥치수가 약간 거칠고, 높이는 DA 특성상 과소평가(꼭대기 뭉갬+바닥 임계). 정밀 크기가 필요하면 08/09.
- **튜닝**: 물체를 놓침 → `min_height_mm`↓; 잔검출 → `min_area_px`↑ / `min_height_mm`↑.
- 향후: 검출된 물체 위치에만 FastSAM을 가끔 돌려 경계·높이를 보정(정밀+속도 절충).